In [ ]:
#***************************************************************************************************#
# CODE DEVELOPED BY CHISOMO DAKA - MARCH, 2025 (WITS UNIVERSITY - NANO-SCALE TRANSPORT PHYSICS LAB) #
#***************************************************************************************************#
import time
import json
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import ZFeatureMap, ZZFeatureMap
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import BaseEstimatorV2, StatevectorEstimator as BaseEstimatorV2, Estimator
from qiskit.primitives import BaseSamplerV2, BaseSamplerV1, StatevectorSampler as BaseSamplerV2, BaseSamplerV1, StatevectorSampler
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import Optimize1qGates, Collect2qBlocks, UnitarySynthesis
from qiskit_machine_learning.optimizers import COBYLA, POWELL, NELDER_MEAD #Gradient Free Optimizers
from qiskit_machine_learning.utils import algorithm_globals
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier, VQC
from qiskit_machine_learning.neural_networks import EstimatorQNN, SamplerQNN
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA #Use PCA -> Principal Component Analysis to further reduce feature dimensions (Optional)
#*****************************
algorithm_globals.random_seed = 12345 #Use 12345 as the starting point for all random number generation across algorithms, circuits, and simulators that depend on randomness.

#*********************
#GENERALIZED CONVOLUTIONAL AND POOLING UNITARIES
# We now define a generalized two-qubit convolutional unitary
num_convolutional_params = 2 #params
def conv_circuit(params):
    target = QuantumCircuit(2)
    target.rz(np.pi / 2, 1)
    target.cx(1, 0)
    target.rz((2*params[0]-np.pi/2), 0)
    target.ry((np.pi/2-2*params[1]), 1)
    target.cx(1, 0)
    target.rz(-np.pi / 2, 0)
    return target

# We now define a generalized two-qubit pooling unitary
num_pooling_params = 2 #params
def pool_circuit(params):
    target = QuantumCircuit(2)
    target.rz(-np.pi / 2, 1)
    target.cx(1, 0)
    target.rz((2*params[0]-np.pi/2), 0)
    target.ry((np.pi/2-2*params[1]), 1)
    target.cx(0, 1)
    target.ry(np.pi / 2, 0)
    return target

#GENERALIZED CONVOLUTIONAL LAYER AND POOLING LAYER (with Sources & Sinks)
# We now define a generalized Q-qubit convolutional layer: Q = num_qubits -> even
def conv_layer(num_qubits, param_prefix, iterX):
    qc = QuantumCircuit(num_qubits, name="Convolutional Layer-"+str(iterX))
    qubits = list(range(num_qubits))
    param_index = 0
    params = ParameterVector(param_prefix, length=num_qubits * num_convolutional_params)
    for q1, q2 in zip(qubits[0::2], qubits[1::2]):
        qc = qc.compose(conv_circuit(params[param_index : (param_index + num_convolutional_params)]), [q1, q2])
        qc.barrier()
        param_index += num_convolutional_params
    for q1, q2 in zip(qubits[1::2], qubits[2::2] + [0]):
        qc = qc.compose(conv_circuit(params[param_index : (param_index + num_convolutional_params)]), [q1, q2])
        qc.barrier()
        param_index += num_convolutional_params

    qc_inst = qc.to_instruction()

    qc = QuantumCircuit(num_qubits)
    qc.append(qc_inst, qubits)
    #qc.barrier() # Add barrier after QCNN convolutional layer
    print("The convolutional layer circuit below has", param_index, "parameters")
    return qc

# We now define a generalized Q-qubit pooling layer: Q = num_qubits
def pool_layer(sources, sinks, param_prefix, iterX):
    #num_qubits = num_qbts
    num_qubits = len(sources) + len(sinks)
    print("Pooling qubits:", num_qubits)
    qc = QuantumCircuit(num_qubits, name="Pooling Layer-"+str(iterX))
    param_index = 0
    params = ParameterVector(param_prefix, length=num_qubits // 2 * num_pooling_params)
    for source, sink in zip(sources, sinks):
        qc = qc.compose(pool_circuit(params[param_index : (param_index + num_pooling_params)]), [source, sink])
        qc.barrier()
        param_index += num_pooling_params

    qc_inst = qc.to_instruction()

    qc = QuantumCircuit(num_qubits)
    qc.append(qc_inst, range(num_qubits))
    #qc.barrier()
    return qc

#Create an auto generation function for sources and sinks lists -> pooling layer outputs
def generate_sources_sinks (num_qubits):
    #declare empty sources and sinks lists
    sources, sinks = [],[]
    #Proceed only if the number of qubits is even
    if num_qubits%2 == 0:
        for i in range(num_qubits):
            if i < int(num_qubits/2): 
                sources.append(i) # sources are the first half number of qubits
            else:                     
                sinks.append(i)   # sinks are the last half number of qubits
    #Report error for old number of qubits
    else:
        print("You have provided an old number of qubits!!!")
        #return sources and sinks
    return sources, sinks

#GENARATING MULTINARY IMAGE DATASET (N = num_images_m_m_m_m_m_m)
#Generate Labeled Multinary Pixalated Image Dataset (num_images, num_qubits)
def generate_random_labelled_multinary_image_dataset(num_images, row_pixels):
    images = []  # List to store image data
    labels = []  # List to store image labels
    image_pixels = row_pixels*row_pixels

    # Define RGB values for Red, Green, and Blue
    red_color = np.array([255, 0, 0])  # Red
    green_color = np.array([0, 255, 0])  # Green
    blue_color = np.array([0, 0, 255])  # Blue
    rgb_colors = [red_color, green_color, blue_color]
    
    # Horizontal and Vertical Lines
    hor_array = np.zeros((row_pixels, image_pixels, 3), dtype=int)  # n rows, n × n pixels, 3 channels (RGB)
    ver_array = np.zeros((row_pixels, image_pixels, 3), dtype=int)  # n rows, n × n pixels, 3 channels (RGB)
    #print(hor_array[0])
    hor_line_start_points = [i * row_pixels for i in range(row_pixels)]  # Row start indices in a flattened array
    ver_line_start_points = [j for j in range(row_pixels)]  # Column start indices in a flattened array

    # Populate Horizontal Lines with Random Colors
    for i in hor_line_start_points:
        #print(i)
        #for color in rgb_colors:
        #color_choice = np.random.choice([0, 1, 2])
        color_choice = i % 3 # color_coice E [0, 1,2]
        #if i == 0: print(color)
        for x in range(row_pixels):  # Fill n sequential pixels to form a horizontal line
            row = i // row_pixels  # Calculate row index for the line
            color_position = i + x
            hor_array[row][color_position] = rgb_colors[color_choice]  # Set the pixel to the current color
    #print(hor_array[2])

    # Populate Vertical Lines with Random Colors
    for j in ver_line_start_points:
        #color_choice = np.random.choice([0, 1, 2])
        color_choice = j % 3
        for y in range(row_pixels):  # Fill n sequential pixels to form a vertical line
            row = j
            color_position = j + (row_pixels*y)  # Column index (should stay within bounds)
            ver_array[row][color_position] = rgb_colors[color_choice]  # Set the pixel to the current color
    
    #print(ver_array[2])
    # Generate `num_images` with balanced classes
    x, y = 0, 0  # Track line positions
    for n_img in range(num_images):
        if n_img < num_images // 2:  # First half -> Horizontal Line Images
            # Randomly assign color and class for horizontal lines (labels)
            y = np.random.choice(range(row_pixels)) # random H-Line Image
            if np.any(np.all(hor_array[y] == red_color, axis=1)):
                labels.append([-1,"R"])  # Horizontal Red
            elif np.any(np.all(hor_array[y] == green_color, axis=1)):
                labels.append([-1,"G"])  # Horizontal Green
            else:
                labels.append([-1,"B"])  # Horizontal Blue
                
            images.append(np.array(hor_array[y]))  # Add horizontal line image
            
        else:  # Second half -> Vertical Line Images
            # Randomly assign color and class for vertical lines
            x = np.random.choice(range(row_pixels)) # random V-Line Image
            if np.any(np.all(ver_array[x] == red_color, axis=1)):
                labels.append([+1,"R"])  # Vertical Red
            elif np.any(np.all(ver_array[x] == green_color, axis=1)):
                labels.append([+1,"G"])  # Vertical Green
            else:
                labels.append([+1,"B"])  # Vertical Blue

            images.append(np.array(ver_array[x]))  # Add vertical line image
    
    #print(labels)
    #print(hor_array)
    # Insert Background Noise (prevents memorization)
    #total_images = int(len(images))
    #print(len(images))
    for image in images:  # Iterate over pixels to add random noise
        #print(image)
        for pixel in range(image_pixels):
            if np.array_equal(image[pixel], [0, 0, 0]):  # Modify only background pixels (black pixels)
                noise = np.random.randint(50, 127, 3)  # Random noise for RGB channels in range [0, 127]
                image[pixel] = noise  # Set random noise as background pixel
                
    return images, labels  # Return generated dataset

#Label Color and Image Orientation Dictionary
def detect_color_orientation (label):
    color_code = []
    color_name = ""
    orientation = ""
    #print(label)
    #Detect Orientation
    if label[0] == -1: # Horizontal
        #orientation = "Horizontal"
        orientation = "x"
    else: # Vertical Line
        #orientation = "Vertical"
        orientation = "y"
    # Detect Color -> Code & Name
    if label[1] == "R":
        color_code = [255, 0, 0]
        color_name = "Red"
    elif label[1] == "G":
        color_code = [0, 255, 0]
        color_name = "Green"
    else:
        color_code = [0, 0, 255]
        color_name = "Blue"
    #Return Orientation Color Dictionary
    return {'c_code': color_code, 'c_name': color_name, 'orientation': orientation}

#RGB TO INT DATASET IMAGE CONVERSION
#Forward
def rgb_to_int_dataset(rgb_image_dataset):
    """
    Converts a dataset of RGB images to integer-represented pixels.
    """
    processed_dataset = []
    # Iterate through each image in the dataset
    for image in rgb_image_dataset:
        new_image = []
        # Iterate through each of the 64 pixels in the image
        for pixel in image:
            # Unpack the r, g, b values from the pixel list
            r, g, b = pixel
            
            # Apply the bit-shifting formula to get a unique integer
            integer_value = (r << 16) | (g << 8) | b
            
            # Add the new integer pixel to our new image representation
            new_image.append(integer_value)
        
        # Add the fully processed image to our new dataset
        processed_dataset.append(new_image)
        
    return processed_dataset
#Reverse
def int_to_rgb_dataset(int_image_dataset):
    """
    Converts a dataset of integer-represented pixels back to RGB images.
    """
    rgb_dataset = []
    # Iterate through each image in the integer dataset
    for integer_image in int_image_dataset:
        new_image = []
        # Iterate through each of the 64 integer pixels in the image
        for integer_pixel in integer_image:
            # Apply bit-shifting and masking to extract r, g, b values
            # Extract red: shift right by 16 bits and mask
            r = (integer_pixel >> 16) & 0xFF
            
            # Extract green: shift right by 8 bits and mask
            g = (integer_pixel >> 8) & 0xFF
            
            # Extract blue: no shift needed, just mask
            b = integer_pixel & 0xFF
            
            # Add the [r, g, b] pixel to our new image representation
            new_image.append([r, g, b])
        
        # Add the fully processed image to our new dataset
        rgb_dataset.append(new_image)
        
    return rgb_dataset

#PCA Feature Reduction
def apply_pca_reduction_to_16_features (X, n_components=16, normalize=True, variance_threshold=None):
    """
    Apply Principal Component Analysis (PCA) to reduce the dimensionality of image data
    for quantum or hybrid quantum-classical models such as No-QCNN or VQC.
    """
    # Initialize PCA model
    if variance_threshold is not None:
        pca = PCA(variance_threshold)
    else:
        pca = PCA(n_components=n_components)
    
    # Fit PCA and transform the dataset
    X_reduced = pca.fit_transform(X)
    
    # Normalize each sample to unit length if required
    if normalize:
        norms = np.linalg.norm(X_reduced, axis=1, keepdims=True)
        X_reduced = np.divide(X_reduced, norms, where=norms!=0)
    
    # Print diagnostic info
    print(f"[PCA Reduction] Original shape: {X.shape}")
    print(f"[PCA Reduction] Reduced shape: {X_reduced.shape}")
    print(f"[PCA Reduction] Explained variance ratio: {np.sum(pca.explained_variance_ratio_):.4f}")
    
    return X_reduced #, pca

#INITIALIZE EXPERIMENT VARIABLES
n = 3 ##8x8 Images corresponds to number of conv-pool layers -> l = 2*n - 2
num_conv_pool_layers = 2*n-4#2*n-2 # num conv-pool-layers (based on image square size -> n)
num_initialized_qubits_Q = 2**(n-1) * 2**(n-1) #(Q = 2^n x 2^n)
multinary_image_dataset_size = 100 # size of multinary dataset N
row_pix = 2**n #row = 2^n -> n = 3 
# TR% -> Training Sample Size and TE% -> Testing Sample Size # Vary training-testing dataset ratio: 70:30, 80:20, or 90:10
train_data_size = 0.7 # 70% training data
test_data_size = 0.3 #30% test data

#GENERATE FULL MULTINARY IMAGE DATASET (WITH TRAINING AND TESTING SUB-DATASETS) -> Size N
#Image Parameters

# Generate a pixelated multinary dataset with N-image samples (6 Classes)
images_m, labels_m = generate_random_labelled_multinary_image_dataset(multinary_image_dataset_size, row_pix)
if (len(images_m) == multinary_image_dataset_size and len(labels_m) == multinary_image_dataset_size):
    print("A well balanced Block Matrix Image Dataset with", multinary_image_dataset_size, " images has been created successfully")

# Generate training and testing sub-datasets (images and labels)
train_images_m, test_images_m, train_labels_m, test_labels_m = train_test_split(
    images_m,
    labels_m, 
    train_size=train_data_size,
    test_size=test_data_size, 
    random_state=246, 
    stratify=labels_m #Ensures that both train_labels and test_labels have the same class distribution as original dataset labels
)

#Make Sure You Have a Well Ballanced Multi-Class Training Set
print("Horizontal Training Samples (Red):", train_labels_m.count([-1, "R"]))
print("Horizontal Training Samples (Green):", train_labels_m.count([-1, "G"]))
print("Horizontal Training Samples (Blue):", train_labels_m.count([-1, "B"]))
print("Vertical Training Samples (Red):", train_labels_m.count([+1, "R"]))
print("Vertical Training Samples (Green):", train_labels_m.count([+1, "G"]))
print("Vertical Training Samples (Blue):", train_labels_m.count([+1, "B"]))

#Optional
#VISUALIZE GENERATED DATASET
"""
# Example: Visualize the first 5 images from train_images
num_images_m = int(len(images_m))
# Calculate grid size
grid_size = int(np.ceil(np.sqrt(num_images_m)))  # Rounded up to nearest integer

# Create subplots with grid_size x grid_size dimensions
fig, axes = plt.subplots(grid_size, grid_size, figsize=(14, 14), subplot_kw={"xticks": [], "yticks": []})

# Flatten axes array to easily index the subplots
axes = axes.flatten()

# Plot the images
for i in range(num_images_m):
    axes[i].imshow(images_m[i].reshape(row_pix, row_pix, 3), aspect="equal")
    axes[i].axis('off')  # Hide axes for better clarity
    #Label Image
    this_color = detect_color_orientation(labels_m[i])
    #axes[i].set_title(str(this_color['orientation'])[:3]+'-'+this_color['c_name'])
    axes[i].set_title(str('["'+this_color['orientation'])[:3]+'", "'+this_color['c_name'][:1]+'"]')
# Hide any extra subplots if num_images is less than grid size
for j in range(num_images_m, len(axes)):
    axes[j].axis('off')

plt.subplots_adjust(wspace=0.1, hspace=0.5)
plt.show()
"""

#BUILD THE No-QCNN NEURAL NETWORK FRAMEWORK
#Declare estimator & base estimator v2 (optional -> Qiskit Primitives)
estimatorV1 = Estimator()
estimatorV2 = BaseEstimatorV2()
SV_sampler = StatevectorSampler()
#samplerV1 = BaseSamplerV1()
samplerV2 = BaseSamplerV2() #BaseSamplerV2, BaseSamplerV1, StatevectorSampler
base_pass_manager = PassManager([Collect2qBlocks(),UnitarySynthesis()])
#z_feature_map = ZFeatureMap(num_initialized_qubits_Q) #No-QCNN data Encoding feature map -> pixel wise encoding
#Declare the ZZFeatureMap -> Linear Entanglement
zz_feature_map = ZZFeatureMap(num_initialized_qubits_Q, reps=1, insert_barriers=True, entanglement="linear") #No-QCNN data Encoding feature map -> pixel wise encoding

#Generating l = n Convolutional-Pooling Layers (num_qubits should be powers of 2 -> special even numbers -> n = 2++)
def generate_conv_pool_layers(num_qbts, num_layers_l):
    #Declare No-QCNN Convolutional-Pooling Circuit Layers
    conv_pool_qc_layers = QuantumCircuit(num_qbts, name="Conv_Pool_Circuit_Layers")
    # number of layers l = n
    #n = int(np.log2(num_qbts)) ->
    No_QCNN_layers = num_layers_l# N = n (i.e. num layers l = n, for num_qubits Q = 2^n x 2^n)
    for i in range(0, No_QCNN_layers):
        if i == 0:
            start_point = 0
            remainder = 0
        else:
            remainder = num_qbts - start_point
            start_point = start_point + int (remainder/2)

        print("Start point:", start_point)
        # n-th Convolutional Layer
        conv_pool_qc_layers.compose(conv_layer(int(num_qbts/(2**i)), "c"+str(i+1), i+1), list(range(start_point, num_qbts)), inplace=True, inline_captures=False)
        # n-th Pooling Layer
        #sources, sinks = generate_sources_sinks (int(num_qbts/(2**i)))
        print('Sources + Sinks :', len(list(range(0,int(num_qbts/(2**(i+1)))))) + len(list(range(int(num_qbts/(2**(i+1))), int(num_qbts/((2**(i))))))))
        conv_pool_qc_layers.compose(pool_layer(list(range(0,int(num_qbts/(2**(i+1))))), list(range(int(num_qbts/(2**(i+1))), int(num_qbts/((2**(i)))))), "p"+str(i+1), i+1), list(range(start_point, num_qbts)), inplace=True)
        #barrier after each (conv+pool) layer
        conv_pool_qc_layers.barrier()
    #Return Total Successive No-QCNN Convolutional-Pooling Circuit Layers
    return conv_pool_qc_layers

#Assign No-QCNN Convolutional-Pooling Circuit Layers (with l = n)
No_QCNN_conv_pool_layers = generate_conv_pool_layers(num_initialized_qubits_Q, num_conv_pool_layers)

#Generate No-QCNN Feature Map -> Combining the ZZFeatureMap and No-QCNN Convolutional-Pooling Circuit Layers
def generate_feature_map_observable(num_qbts, conv_pool_layers, FM):
    #Declare No-QCNN Feature Map Circuit
    FM_circuit = QuantumCircuit(num_qbts)
    FM_circuit.compose(FM, range(num_qbts), inplace=True)
    FM_circuit.barrier() #barrier after feature map (for visual display only)
    FM_circuit.compose(conv_pool_layers, range(num_qbts), inplace=True)
    #Declare No-QCNN Observable
    FM_observable = SparsePauliOp.from_list([("I" * (num_qbts-4) + "ZZZZ", 1)]) # No-QCNN Observable in Z-basis -> Fully connected circuit on last 4 Q-bits
    #Return No-QCNN Feature Map and Observable
    return FM_circuit, FM_observable

#Initialize No-QCNN Feature Map and Observable (num_qubits, conv-pool-layers, zzfeaturemap)
No_QCNN_Feature_Map, No_QCNN_Observable = generate_feature_map_observable(num_initialized_qubits_Q, No_QCNN_conv_pool_layers, zz_feature_map)

#Multinary No-QCNN Neural Network
def generate_multinary_No_QCNN_neural_network(No_QCNN_FM, No_QCNN_obs, zz_FM, conv_pool_layers):
    multinary_neural_network = EstimatorQNN(
        circuit=No_QCNN_FM.decompose(), # we decompose the No-QCNN Feature Map circuit to avoid additional data copying
        pass_manager = None, #base_pass_manager #optional pass_manager instance for primitives that require transpilation (cloud)
        observables=No_QCNN_obs, # defines the expectation value computation basis
        input_params=zz_FM.parameters, # list of quantum circuit parameters that should be treated as “network inputs”
        weight_params=conv_pool_layers.parameters, # list of quantum circuit parameters that should be treated as “network weights”
        estimator=estimatorV2, #optional primitive instance
    )
    return multinary_neural_network

No_QCNN_neural_network_sampler=SamplerQNN(
    circuit=No_QCNN_Feature_Map.decompose(),
    input_params=zz_feature_map.parameters, # list of quantum circuit parameters that should be treated as “network inputs”
    weight_params=No_QCNN_conv_pool_layers.parameters, # list of quantum circuit parameters that should be treated as “network weights”
    sampler=SV_sampler
)

#Initialize the Fully Composed No-QCNN Multinary Neural Network Framework
No_QCNN_multinary_neural_network = generate_multinary_No_QCNN_neural_network(No_QCNN_Feature_Map,No_QCNN_Observable,zz_feature_map,No_QCNN_conv_pool_layers)

# =============================================
#   No-QCNN TRAINING EXPERIMENT CONFIGURATION
# =============================================
#SETUP THE MULTINARY CLASSIFICATION EXPERIMENT - 6 CLASSES
max_opt_iter = 100
num_images_m = int(len(images_m))
plt.rcParams["figure.figsize"] = (14, 8)

""""""
#
# Convert labels to range [0, 5] - 6 labels
train_labels_m = np.array([
    0 if np.array_equal(tr_label, [-1, "R"]) else
    1 if np.array_equal(tr_label, [-1, "G"]) else
    2 if np.array_equal(tr_label, [-1, "B"]) else
    3 if np.array_equal(tr_label, [+1, "R"]) else
    4 if np.array_equal(tr_label, [+1, "G"]) else
    5 #if np.array_equal(tr_label, [+1, "B"])
    for tr_label in train_labels_m
])
test_labels_m = np.array([
    0 if np.array_equal(te_label, [-1, "R"]) else
    1 if np.array_equal(te_label, [-1, "G"]) else
    2 if np.array_equal(te_label, [-1, "B"]) else
    3 if np.array_equal(te_label, [+1, "R"]) else
    4 if np.array_equal(te_label, [+1, "G"]) else
    5 #if np.array_equal(te_label, [+1, "B"])
    for te_label in test_labels_m
])

#Training Image Dataset and Labels Arrays
#train_images_array = np.asarray(train_images_m)
train_images_array = np.asarray(rgb_to_int_dataset(train_images_m))
train_labels_array = np.asarray(train_labels_m)

#Testing Image Dataset and Labels Arrays
#test_images_array = np.asarray(test_images_m)
test_images_array = np.asarray(rgb_to_int_dataset(test_images_m))
test_labels_array = np.asarray(test_labels_m)

#Apply PCA Dimension Reduction to 16 Features on Train and Test Images
train_images_array_16_F =apply_pca_reduction_to_16_features(train_images_array)
test_images_array_16_F =apply_pca_reduction_to_16_features(test_images_array)

#Intitialize Variables
#weights_vals = []
#objective_func_vals = []
#max_opt_iter = 100 #Maximum ieterations
#epoch_list = [5, 10, 15, 20] #list of training epochs

# =============================
#   TRAINING LOGGING STRUCTURE
# =============================
training_logs_m = {
    "iteration": [],
    "objective": [],
    "weights": [],
    "elapsed_time": [],
    "avg_weight": [],
    "min_weight": [],
    "max_weight": [],
    "approx_accuracy": [],  # pseudo accuracy estimate
}

start_time = time.time()

# =============================
#   CALLBACK FUNCTION
# =============================
def callback_graphs_m(weights, obj_func_eval):
    """Callback that logs optimizer progress and plots multiple metrics safely."""
    current_iter = len(training_logs_m["iteration"])
    elapsed_time = time.time() - start_time

    # Log numerical metrics
    training_logs_m["iteration"].append(current_iter)
    training_logs_m["objective"].append(obj_func_eval)
    training_logs_m["weights"].append(weights)
    training_logs_m["elapsed_time"].append(elapsed_time)
    training_logs_m["avg_weight"].append(np.mean(weights))
    training_logs_m["min_weight"].append(np.min(weights))
    training_logs_m["max_weight"].append(np.max(weights))

    # --- Approximate accuracy (safe placeholder, avoids 'not fitted' error) ---
    # We can't call classifier.score() yet, so we'll simulate an accuracy proxy
    # from relative objective improvement or weight stability.
    if current_iter > 0:
        prev_obj = training_logs_m["objective"][-2]
        delta_obj = abs(prev_obj - obj_func_eval)
        pseudo_acc = np.exp(-delta_obj) * 100  # heuristic proxy (decays with loss)
    else:
        pseudo_acc = 0.0
    training_logs_m["approx_accuracy"].append(pseudo_acc)

    # --- Dynamic Multi-Metric Plotting ---
    clear_output(wait=True)
    plt.style.use('default')
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))

    objective_error = np.array(training_logs_m.get("objective_error", [0.075]*len(training_logs_m["objective"])))

    iterations = training_logs_m["iteration"]
    objective_values = training_logs_m["objective"]

    # Plot main line
    axs[0, 0].plot(iterations, objective_values, '-o', color='royalblue', label="Objective Value")

    # Plot error bounds
    axs[0, 0].fill_between(iterations,
                        np.array(objective_values) - objective_error,
                        np.array(objective_values) + objective_error,
                        color='royalblue', alpha=0.2)

    # Labels, title, legend, grid
    axs[0, 0].set_title("Objective Function vs Iteration")
    axs[0, 0].set_xlabel("Iteration")
    axs[0, 0].set_ylabel("Objective Value")
    axs[0, 0].legend()
    # Set grid at every 20 iterations
    #axs[0, 0].xaxis.set_major_locator(MultipleLocator(20))  # vertical grid every 10 iterations
    #axs[0, 0].grid(True, linestyle='--', alpha=0.5)  # dashed grid
    # Add subplot label (a)
    axs[0, 0].text(
        0, 1.075, '(a)', 
        transform=axs[0, 0].transAxes,  # relative to axes
        fontsize=14, 
        fontweight='normal', 
        va='top', 
        ha='right'
    )

    # Approximate Accuracy
    axs[0, 1].plot(training_logs_m["iteration"], training_logs_m["approx_accuracy"],
                    '-*', color='seagreen', label="Approx. Accuracy")
    axs[0, 1].set_title("Approximate Accuracy Progression")
    axs[0, 1].set_xlabel("Iteration")
    axs[0, 1].set_ylabel("Accuracy Proxy (%)")
    axs[0, 1].legend()
    # Set grid at every 20 iterations
    #axs[0, 1].xaxis.set_major_locator(MultipleLocator(20))  # vertical grid every 10 iterations
    #axs[0, 1].grid(True, linestyle='--', alpha=0.5)  # dashed grid
    # Add subplot label (b)
    axs[0, 1].text(
        0, 1.075, '(b)', 
        transform=axs[0, 1].transAxes,  # relative to axes
        fontsize=14, 
        fontweight='normal', 
        va='top', 
        ha='right'
    )

    # Weight Evolution
    axs[1, 0].plot(training_logs_m["iteration"], training_logs_m["avg_weight"],
                    label="Average Weight", color='purple')
    axs[1, 0].fill_between(
        training_logs_m["iteration"],
        training_logs_m["min_weight"],
        training_logs_m["max_weight"],
        alpha=0.25, color='violet', label="Min–Max Range"
    )
    axs[1, 0].set_title("Weight Evolution Across Iterations")
    axs[1, 0].set_xlabel("Iteration")
    axs[1, 0].set_ylabel("Weight Magnitude")
    axs[1, 0].legend()
    # Set grid at every 20 iterations
    #axs[1, 0].xaxis.set_major_locator(MultipleLocator(20))  # vertical grid every 10 iterations
    #axs[1, 0].grid(True, linestyle='--', alpha=0.5)  # dashed grid
    # Add subplot label (c)
    axs[1, 0].text(
        0, 1.075, '(c)', 
        transform=axs[1, 0].transAxes,  # relative to axes
        fontsize=14, 
        fontweight='normal', 
        va='top', 
        ha='right'
    )

    # Elapsed Time
    axs[1, 1].plot(training_logs_m["iteration"], training_logs_m["elapsed_time"],
                    '--d', markevery=4, color='darkred', label="Cum. Training Time")
    axs[1, 1].set_title("Cumulative Training Time")
    axs[1, 1].set_xlabel("Iteration")
    axs[1, 1].set_ylabel("Elapsed Time (s)")
    axs[1, 1].legend()
    # Set grid at every 20 iterations
    #axs[1, 1].xaxis.set_major_locator(MultipleLocator(20))  # vertical grid every 10 iterations
    #axs[1, 1].grid(True, linestyle='--', alpha=0.5)  # dashed grid
    # Add subplot label (d)
    axs[1, 1].text(
        0, 1.075, '(d)', 
        transform=axs[1, 1].transAxes,  # relative to axes
        fontsize=14, 
        fontweight='normal', 
        va='top', 
        ha='right'
    )

    plt.tight_layout()
    plt.show()


# =============================
#   INITIALIZE THE VQC CLASSIFIER
# =============================
No_QCNN_vqc_classifier = VQC(
    num_qubits=num_initialized_qubits_Q,
    feature_map=zz_feature_map,
    ansatz=No_QCNN_conv_pool_layers,
    optimizer=COBYLA(maxiter=max_opt_iter),
    output_shape=6,  # Multinary classification (6 Classes)
    sampler=SV_sampler,
    loss="cross_entropy",
    warm_start=False,
    initial_point=None,
    callback=callback_graphs_m
)

# =============================
#   TRAIN THE MODEL
# =============================
print("Trainign Images: ", len(train_images_array_16_F))
print("Training Labels: ", len(train_labels_array))
print("Training No-QCNN VQC Classifier...")
No_QCNN_vqc_classifier.fit(train_images_array_16_F, train_labels_array)

# =============================
#   FINAL EVALUATION
# =============================

train_accuracy = No_QCNN_vqc_classifier.score(train_images_array_16_F, train_labels_array)
print(f"\nTraining "+str(num_images_m)+ " multinary images complete!")
print(f"Final Training Accuracy: {100 * train_accuracy:.2f}%")
print(f"Total Iterations: {len(training_logs_m['iteration'])}")
print(f"Total Training Time: {training_logs_m['elapsed_time'][-1]:.2f} seconds")
